### Take Luhman (2022a, b) tables and determine the number of sources in Lupus and UpperSco w/ discs in SpT ranges.

In [1]:
import re
import numpy  as np
import pandas as pd
pd.set_option('display.max_columns', None)

`Identify spectral type ranges:` Here I'm following Table 6 of Luhman&Esplin (2020) and Table 3 of Luhman (2022b).

In [2]:
maskM0G2 = ["G2", "G3", "G4", "G5", "G6", "G7", "G8", "G9", "K0", "K1", "K2", "K3", "K4", "K5", "K6", "K7", "K8", "K9"] #0.70-1.40 Msun
maskM4M0 = ["M0", "M1", "M2", "M3"] #0.20-0.70 Msun
maskM8M4 = ["M4", "M5", "M6", "M7"] #0.04-0.20 Msun

`Task #1:` Number of stars in Lupus and Upper Sco according to Luhman (kinematically and kinematically + astrometrically).

In [3]:
# Load Luhman22a's table:
luhman_stars = pd.read_fwf(
    "table1_luhman2022a_orig.dat",
    names  = [
        "SourceID_GaiaDR3", "SourceID_GaiaDR2", "NameOther",
        "RAdeg", "DEdeg",
        "SpType", "r_SpType", "AdoptSpType",
        "EWLi", "r_EWLi",
        "pmRA", "e_pmRA", "pmDec", "e_pmDec", "plx", "e_plx",
        "r_med_geo", "r_lo_geo", "r_hi_geo",
        "RVel", "e_RVel", "r_RVel",
        "U U", "e_U", "V V", "e_V", "W W", "e_W",
        "Gmag G", "e_Gmag", "GBPmag", "e_GBPmag", "GRPmag", "e_GRPmag",
        "RUWE", "kin", "pos",
        "2MASS", "SourceID_2MASS", "2MASS_angsep", "2MASS_closest",
        "WISE", "SourceID_WISE", "WISE_angsep", "WISE_closest"
    ],
    skiprows = 273,
)
print(len(luhman_stars))
# Numbers match: 10509, yes!
print(len(luhman_stars[luhman_stars["AdoptSpType"].notna()])/len(luhman_stars))
# Only 30% of stars have info on SpT...

10509
0.3015510514796841


`Task #0.1:` Update all the adopted SpT in such a way to have those from the PPVII chapter table and Aaron Empey's 2026 paper when possible.

In [4]:
# Load your extended version of the table:
uppersco_mine = pd.read_csv(
    "uppersco_vs_polnitzky.dat",
    names  = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "discClassC25", "discClassL22", "COsizeS21?", "comments"
    ],
    sep      = r"\s+",
    skiprows = 1
)
print(len(uppersco_mine))
# Numbers match: 284.

284


In [5]:
# Make sure the two cols have the same dtype, otherwise merging won't work!
luhman_stars["SourceID_GaiaDR3"] = luhman_stars["SourceID_GaiaDR3"].astype("string")
uppersco_mine["SourceID_GaiaDR3"] = uppersco_mine["SourceID_GaiaDR3"].astype("string")

# Merge and combine the SpT columns, then drop the added one.
luhman_stars = luhman_stars.merge(
    uppersco_mine[["SourceID_GaiaDR3", "SpT_E26"]],
    on  = "SourceID_GaiaDR3",
    how = "left",
)
luhman_stars["AdoptSpType"] = luhman_stars["SpT_E26"].combine_first(luhman_stars["AdoptSpType"])

luhman_stars = luhman_stars.drop(columns="SpT_E26")
# luhman_stars[luhman_stars["SourceID_GaiaDR3"]==6209142346187345024]

In [6]:
# Load your extended version of the table:
lupus_mine = pd.read_csv(
    "lupus_vs_polnitzky.dat",
    skiprows = 48,
    sep      = r'\s+',
    names   = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "mem_T22", "discClassL22", "COsizeS21?", "comments"
    ]
)
print(len(lupus_mine))
# Numbers match: 100, yes!

99


In [7]:
# Make sure the two cols have the same dtype, otherwise merging won't work!
luhman_stars["SourceID_GaiaDR3"] = luhman_stars["SourceID_GaiaDR3"].astype("string")
lupus_mine["SourceID_GaiaDR3"] = lupus_mine["SourceID_GaiaDR3"].astype("string")

# Merge and combine the SpT columns, then drop the added one.
luhman_stars = luhman_stars.merge(
    lupus_mine[["SourceID_GaiaDR3", "SpT_E26"]],
    on  = "SourceID_GaiaDR3",
    how = "left",
)
luhman_stars["AdoptSpType"] = luhman_stars["SpT_E26"].combine_first(luhman_stars["AdoptSpType"])

luhman_stars = luhman_stars.drop(columns="SpT_E26")
# luhman_stars[luhman_stars["SourceID_GaiaDR3"]==6013399894569703040]

`Task 0.1:` Go on with stellar catalogues.

In [8]:
# Kinematic members of Upper Sco:
usco_k_stars = luhman_stars[luhman_stars["kin"].astype(str).str.contains("u", na=False)]
print("Kinematic members of Upper Sco:", len(usco_k_stars))
# 64% of UpperSco stars have info on SpT...
print(
    "Fraction of kinematic members of Upper Sco w/ SpT:", 
    len(usco_k_stars[usco_k_stars["AdoptSpType"].notna()])/len(usco_k_stars))

print("Stars UppeSco M0-G2:", len(usco_k_stars[usco_k_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)]))
print("Stars UppeSco M4-M0:", len(usco_k_stars[usco_k_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)]))
print("Stars UppeSco M8-M4:", len(usco_k_stars[usco_k_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)]))

Kinematic members of Upper Sco: 3103
Fraction of kinematic members of Upper Sco w/ SpT: 0.6416371253625524
Stars UppeSco M0-G2: 185
Stars UppeSco M4-M0: 495
Stars UppeSco M8-M4: 1009


In [9]:
# Kinematic and astrometric members of Upper Sco:
usco_kp_stars = luhman_stars[
    (luhman_stars["kin"].astype(str).str.contains("u", na=False)) & 
    (luhman_stars["pos"].astype(str).str.contains("u", na=False))
]
print("Kinematic and astrometric members of Upper Sco:", len(usco_kp_stars))
# 96% of UpperSco stars have info on SpT...
print(
    "Fraction of kinematic and astrometric members of Upper Sco w/ SpT:", 
    len(usco_kp_stars[usco_kp_stars["AdoptSpType"].notna()])/len(usco_kp_stars))

print("Stars UppeSco M0-G2:", len(usco_kp_stars[usco_kp_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)]))
print("Stars UppeSco M4-M0:", len(usco_kp_stars[usco_kp_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)]))
print("Stars UppeSco M8-M4:", len(usco_kp_stars[usco_kp_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)]))

Kinematic and astrometric members of Upper Sco: 1349
Fraction of kinematic and astrometric members of Upper Sco w/ SpT: 0.9607116382505559
Stars UppeSco M0-G2: 78
Stars UppeSco M4-M0: 338
Stars UppeSco M8-M4: 691


In [10]:
# Kinematic members of Lupus:
lupo_k_stars = luhman_stars[luhman_stars["kin"].astype(str).str.contains("l", na=False)]
print("Kinematic members of Lupus:", len(lupo_k_stars))
# 54% of Lupus stars have info on SpT...
print(
    "Fraction of kinematic members of Lupus w/ SpT:", 
    len(lupo_k_stars[lupo_k_stars["AdoptSpType"].notna()])/len(lupo_k_stars))

print("Stars Lupus M0-G2:", len(lupo_k_stars[lupo_k_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)]))
print("Stars Lupus M4-M0:", len(lupo_k_stars[lupo_k_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)]))
print("Stars Lupus M8-M4:", len(lupo_k_stars[lupo_k_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)]))

Kinematic members of Lupus: 916
Fraction of kinematic members of Lupus w/ SpT: 0.5447598253275109
Stars Lupus M0-G2: 54
Stars Lupus M4-M0: 87
Stars Lupus M8-M4: 283


In [11]:
# Kinematic and astrometric members of Lupus:
lupo_kp_stars = luhman_stars[
    (luhman_stars["kin"].astype(str).str.contains("l", na=False)) & 
    (luhman_stars["pos"].astype(str).str.contains("l", na=False))
]
print("Kinematic and astrometric members of Lupus:", len(lupo_kp_stars))
# 98% of Lupus stars have info on SpT...
print(
    "Fraction of kinematic and astrometric members of Lupus w/ SpT:", 
    len(lupo_kp_stars[lupo_kp_stars["AdoptSpType"].notna()])/len(lupo_kp_stars))

print("Stars Lupus M0-G2:", len(lupo_kp_stars[lupo_kp_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)]))
print("Stars Lupus M4-M0:", len(lupo_kp_stars[lupo_kp_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)]))
print("Stars Lupus M8-M4:", len(lupo_kp_stars[lupo_kp_stars["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)]))

Kinematic and astrometric members of Lupus: 120
Fraction of kinematic and astrometric members of Lupus w/ SpT: 0.9833333333333333
Stars Lupus M0-G2: 17
Stars Lupus M4-M0: 27
Stars Lupus M8-M4: 66


`Task #2:` Number of stars in Lupus and Upper Sco according to Luhman (kinematically and kinematically + astrometrically) with a counterpart in WISEA.

In [12]:
# Load Luhman22b's table:
luhman_discs = pd.read_fwf(
    "table1_luhman2022b_orig.dat",
    names  = [
        "WISE", "SourceID_WISE_discs", "RAdeg","DEdeg", "2MASS", "SourceID_2MASS_discs", "2MASS_angsep",
        "Jmag 2MASS", "e_Jmag", "Hmag 2MASS", "e_Hmag", "Ksmag 2MASS", "e_Ksmag",
        "W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "f_W3mag", "W4mag", "e_W4mag", "f_W4mag",
        "ExcW2 Excess present in W2?", "ExcW3 Excess present in W3?", "ExcW4 Excess present in W4?", "DiskType Disk Typec"
    ],
    skiprows = 45,
)
print(len(luhman_discs))
# Numbers match: 9675, yes!

9675


In [13]:
# matched = luhman_discs.merge(
#     luhman_stars[["SourceID_2MASS", "2MASS_closest", "SourceID_WISE", "WISE_closest", "SourceID_GaiaDR3", "kin", "pos", "AdoptSpType"]],
#     on  = "SourceID_2MASS",
#     how = "left"
# )
# print(len(matched)) #17238
# This crazy number is most likely driven by the presence of NaNs in both columns. 
# Table1 of Luhman (2022b) is built based on WISE, thus, sources with 2MASS in Table1 of Luhman (2022a) but not WISE are not included.
# You have to match on WISE!

# matched = luhman_discs.merge(
#     luhman_stars[["SourceID_WISE", "WISE_closest", "SourceID_2MASS", "2MASS_closest", "SourceID_GaiaDR3", "kin", "pos", "AdoptSpType"]],
#     on  = "SourceID_WISE",
#     how = "left"
# )
# print(len(matched)) #10020
# This is also wrong because often only a single WISE ID - 2MASS ID combination is present in Table1 of Luhman (2022b) 
# while multiple are in Table1 of Luhman (2022a). Matching just by WISE will include both.
# There are 345 duplicated sources (i.e., more Gaia sources are associated to the same WISE ref ID).

In [14]:
# Remove those objects with no WISE counterpart in Luhman (2022a):
luhman_stars_match = luhman_stars.dropna(subset=["SourceID_WISE"])

# Match to add Gaia IDs, kin, pos info:
matched = luhman_discs.merge(
    luhman_stars_match[["SourceID_WISE", "WISE_closest", "SourceID_2MASS", "2MASS_closest", "SourceID_GaiaDR3", "kin", "pos", "AdoptSpType"]],
    left_on  = ["SourceID_WISE_discs", "SourceID_2MASS_discs"],
    right_on = ["SourceID_WISE", "SourceID_2MASS"],
    how      = "left"
)
print(len(matched)) #9970

# For some weird reason there are some WISE IDs in Table 1 of Luhman (2022b) with no counterpart in Luhman (2022a)...
# These are matched to nan nan and I'll drop them since have no association with a Gaia source/kinpos info.
# WISEA  J121959.36-501839.6  184.9973407 -50.3110062 2MASS J12195937-5018380
# is present in Luhman (2022a) as
# WISEA  J121959.36-501839.6  184.9973407 -50.3110062 2MASS J12195938-5018404
# and I updated it accordingly in Luhman (2022b)
num_dropped = matched[["SourceID_WISE","SourceID_2MASS"]].isna().all(axis=1).sum()
print("WISE IDs in Table 1 of Luhman (2022b) with no counterpart in Table 1 of Luhman (2022a):", num_dropped)

matched = matched.dropna(subset=["SourceID_WISE","SourceID_2MASS"],how="all")
print(len(matched)) #9955
print(len(matched[matched["AdoptSpType"].notna()])/len(matched))
# Only 32% of stars have info on SpT...

# Duplicates?
dups = matched[matched["SourceID_WISE"].duplicated(keep=False)]
print(len(np.unique(dups["SourceID_WISE"].tolist())))
# There are 295 duplicated sources (i.e., more Gaia sources are associated to the same WISE ref ID).
# Those share the same WISE ID - 2MASS ID combination is present in Table1 of Luhman (2022b)

# Perfect 'cause 9970 - 295 is 9675, i.e., the size of Table 1 of Luhman (2022b).
# with open("matched.dat", "w") as f:
#     f.write(matched.to_string(index=False))

9970
WISE IDs in Table 1 of Luhman (2022b) with no counterpart in Table 1 of Luhman (2022a): 15
9955
0.3136112506278252
295


`Task #3:` Split them into disc-bearing and disc-less.

In [15]:
# Discless stars:
excluded_class = ["debris/ev trans", "debris/ev trans?", "debris/ev-trans", "debris/ev-trans?", "debris", "III", "III?", "Be", "Be?", "-"]

In [16]:
# The following Lupus targets are debris/evolved tr in Luhman (2022b) but have detected continuum/CO and resolved gas with ALMA. 
# Will update them to count as full discs (just to be properly considered in the counting)
# 2MASS J15580252-3736026 WISEA J155802.50-373602.9
# 2MASS J16071159-3903475 WISEA J160711.59-390347.7
# 2MASS J16085468-3937431 WISEA J160854.67-393743.3
# 2MASS J16083733-3923109 WISEA J160854.67-393743.3

print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J15580252-3736026')&(matched["SourceID_WISE"]=='J155802.50-373602.9')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16071159-3903475')&(matched["SourceID_WISE"]=='J160711.59-390347.7')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16085468-3937431')&(matched["SourceID_WISE"]=='J160854.67-393743.3')])
# print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16083733-3923109')])

mask = (
    ((matched["SourceID_2MASS"] == "J15580252-3736026") & (matched["SourceID_WISE"] == "J155802.50-373602.9")) |
    ((matched["SourceID_2MASS"] == "J16071159-3903475") & (matched["SourceID_WISE"] == "J160711.59-390347.7")) |
    ((matched["SourceID_2MASS"] == "J16085468-3937431") & (matched["SourceID_WISE"] == "J160854.67-393743.3"))
    # ((matched["SourceID_2MASS"] == "J16083733-3923109")                                                      )
)
# Inspect the matched rows:
# print(matched.loc[mask])     
matched.loc[mask,"DiskType Disk Typec"] = "full_riclassified"

print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J15580252-3736026')&(matched["SourceID_WISE"]=='J155802.50-373602.9')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16071159-3903475')&(matched["SourceID_WISE"]=='J160711.59-390347.7')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16085468-3937431')&(matched["SourceID_WISE"]=='J160854.67-393743.3')])
# print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16083733-3923109')])

5881    debris/ev-trans
Name: DiskType Disk Typec, dtype: object
6579    debris/ev-trans
Name: DiskType Disk Typec, dtype: object
6762    debris/ev-trans
Name: DiskType Disk Typec, dtype: object
5881    full_riclassified
Name: DiskType Disk Typec, dtype: object
6579    full_riclassified
Name: DiskType Disk Typec, dtype: object
6762    full_riclassified
Name: DiskType Disk Typec, dtype: object


In [17]:
# UpperSco kinematics
matched_usco_k  = matched[ matched["kin"].astype(str).str.contains("u", na=False) ]
print(len(matched_usco_k))
# UpperSco kinematics and astrometry
matched_usco_kp = matched[(matched["kin"].astype(str).str.contains("u", na=False)) & (matched["pos"].astype(str).str.contains("u", na=False))]
print(len(matched_usco_kp))

# Lupus kinematics
matched_lupo_k  = matched[ matched["kin"].astype(str).str.contains("l", na=False) ]
print(len(matched_lupo_k))
# Lupus kinematics and astrometry
matched_lupo_kp = matched[(matched["kin"].astype(str).str.contains("l", na=False)) & (matched["pos"].astype(str).str.contains("l", na=False))]
print(len(matched_lupo_kp))

2962
1323
860
119


In [18]:
# The membership of 13 Upper Sco targets was re-evaluated by Carpenter et al. (2025) based on their ALMA data.
# The following are likely to be background objects and will be removed:
excludeobjs2MASS = ["J15442550-2126408", "J15564244-2039339", "J15594231-2945495", "J16044876-1748393", "J16050647-1734020", "J16153220-2010236", "J16185382-2053182", "J15524851-2845369", "J15594460-2155250", "J16052556-2035397", "J16064385-1908056", "J16120505-2043404", "J16185277-2259537"]

pK = matched_usco_k[matched_usco_k["SourceID_2MASS"].isin(excludeobjs2MASS)]
print(pK["SourceID_2MASS"].tolist(), pK["SourceID_WISE"].tolist())
pKP = matched_usco_kp[matched_usco_kp["SourceID_2MASS"].isin(excludeobjs2MASS)]
print(pKP["SourceID_2MASS"].tolist(), pKP["SourceID_WISE"].tolist())

mask            = matched_usco_k["SourceID_2MASS"].isin(excludeobjs2MASS)
matched_usco_k  = matched_usco_k[~mask].copy()
print(len(matched_usco_k["SourceID_2MASS"]))

mask            = matched_usco_kp["SourceID_2MASS"].isin(excludeobjs2MASS)
matched_usco_kp = matched_usco_kp[~mask].copy()
print(len(matched_usco_kp["SourceID_2MASS"]))

['J15564244-2039339', 'J15564244-2039339', 'J16044876-1748393', 'J16044876-1748393', 'J16050647-1734020', 'J16052556-2035397', 'J16064385-1908056', 'J16185277-2259537'] ['J155642.45-203934.1', 'J155642.45-203934.1', 'J160448.71-174838.9', 'J160448.71-174838.9', 'J160506.45-173402.5', 'J160525.55-203539.9', 'J160643.85-190805.8', 'J161852.77-225954.0']
['J15564244-2039339', 'J15564244-2039339', 'J16052556-2035397', 'J16064385-1908056', 'J16185277-2259537'] ['J155642.45-203934.1', 'J155642.45-203934.1', 'J160525.55-203539.9', 'J160643.85-190805.8', 'J161852.77-225954.0']
2954
1318


In [19]:
# Single targets:
unique_usco_k = matched_usco_k[~matched_usco_k["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_usco_k   = matched_usco_k[ matched_usco_k["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_usco_k)+len(dups_usco_k)}")
print(f"Length duplicates: {len(np.unique(dups_usco_k["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_usco_k_discless    = dups_usco_k[ dups_usco_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_usco_k_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_usco_k_discbearing = dups_usco_k[~dups_usco_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_usco_k_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_usco_k_discbearing[dups_usco_k_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
usco_k_both = pd.concat([unique_usco_k, dups_usco_k_discless, dups_usco_k_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(usco_k_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
usco_k_sing = pd.concat([unique_usco_k, dups_usco_k_discless, dups_usco_k_discbearing[dups_usco_k_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(usco_k_sing)}")

# Select disc-bearing stars in each catalogue
usco_kDB_both = usco_k_both[~usco_k_both["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs both  : {len(usco_kDB_both)}")
usco_kDB_sing = usco_k_sing[~usco_k_sing["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs single: {len(usco_kDB_sing)}")

Check that length unique+duplicates matches table: 2954
Length duplicates: 74
Length duplicates disc less   : 61
Length duplicates disc bearing: 13
Length duplicates closest disc bearing: 13
Number of stars both  : 2954
Number of stars single: 2941
Number of stars w/ discs both  : 547
Number of stars w/ discs single: 534


In [20]:
print(
    "Number of UpperSco stars in WISEA with SpT M0-G2 (both):", 
    len(usco_k_both[usco_k_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (both):", 
    len(usco_kDB_both[usco_kDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of UpperSco stars in WISEA with SpT M0-G2 (single):", 
    len(usco_k_sing[usco_k_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (single):", 
    len(usco_kDB_sing[usco_kDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)

print(
    "Number of UpperSco stars in WISEA with SpT M4-G0 (both):", 
    len(usco_k_both[usco_k_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (both):", 
    len(usco_kDB_both[usco_kDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of UpperSco stars in WISEA with SpT M4-G0 (single):", 
    len(usco_k_sing[usco_k_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (single):", 
    len(usco_kDB_sing[usco_kDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)

print(
    "Number of UpperSco stars in WISEA with SpT M8-M4 (both):", 
    len(usco_k_both[usco_k_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (both):", 
    len(usco_kDB_both[usco_kDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of UpperSco stars in WISEA with SpT M8-M4 (single):", 
    len(usco_k_sing[usco_k_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (single):", 
    len(usco_kDB_sing[usco_kDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)

Number of UpperSco stars in WISEA with SpT M0-G2 (both): 182
Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (both): 53
Number of UpperSco stars in WISEA with SpT M0-G2 (single): 182
Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (single): 53
Number of UpperSco stars in WISEA with SpT M4-G0 (both): 491
Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (both): 100
Number of UpperSco stars in WISEA with SpT M4-G0 (single): 490
Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (single): 99
Number of UpperSco stars in WISEA with SpT M8-M4 (both): 987
Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (both): 221
Number of UpperSco stars in WISEA with SpT M8-M4 (single): 983
Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (single): 217


In [21]:
# Single targets:
unique_usco_kp = matched_usco_kp[~matched_usco_kp["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_usco_kp   = matched_usco_kp[ matched_usco_kp["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_usco_kp)+len(dups_usco_kp)}")
print(f"Length duplicates: {len(np.unique(dups_usco_kp["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_usco_kp_discless    = dups_usco_kp[ dups_usco_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_usco_kp_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_usco_kp_discbearing = dups_usco_kp[~dups_usco_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_usco_kp_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_usco_kp_discbearing[dups_usco_kp_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
usco_kp_both = pd.concat([unique_usco_kp, dups_usco_kp_discless, dups_usco_kp_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(usco_kp_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
usco_kp_sing = pd.concat([unique_usco_kp, dups_usco_kp_discless, dups_usco_kp_discbearing[dups_usco_kp_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(usco_kp_sing)}")

# Select disc-bearing stars in each catalogue
usco_kpDB_both = usco_kp_both[~usco_kp_both["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs both  : {len(usco_kpDB_both)}")
usco_kpDB_sing = usco_kp_sing[~usco_kp_sing["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs single: {len(usco_kpDB_sing)}")

Check that length unique+duplicates matches table: 1318
Length duplicates: 29
Length duplicates disc less   : 24
Length duplicates disc bearing: 5
Length duplicates closest disc bearing: 5
Number of stars both  : 1318
Number of stars single: 1313
Number of stars w/ discs both  : 257
Number of stars w/ discs single: 252


In [22]:
print(
    "Number of UpperSco stars in WISEA with SpT M0-G2 (both):", 
    len(usco_kp_both[usco_kp_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (both):", 
    len(usco_kpDB_both[usco_kpDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of UpperSco stars in WISEA with SpT M0-G2 (single):", 
    len(usco_kp_sing[usco_kp_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (single):", 
    len(usco_kpDB_sing[usco_kpDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)

print(
    "Number of UpperSco stars in WISEA with SpT M4-G0 (both):", 
    len(usco_kp_both[usco_kp_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (both):", 
    len(usco_kpDB_both[usco_kpDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of UpperSco stars in WISEA with SpT M4-G0 (single):", 
    len(usco_kp_sing[usco_kp_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (single):", 
    len(usco_kpDB_sing[usco_kpDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)

print(
    "Number of UpperSco stars in WISEA with SpT M8-M4 (both):", 
    len(usco_kp_both[usco_kp_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (both):", 
    len(usco_kpDB_both[usco_kpDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of UpperSco stars in WISEA with SpT M8-M4 (single):", 
    len(usco_kp_sing[usco_kp_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (single):", 
    len(usco_kpDB_sing[usco_kpDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)

Number of UpperSco stars in WISEA with SpT M0-G2 (both): 76
Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (both): 11
Number of UpperSco stars in WISEA with SpT M0-G2 (single): 76
Number of UpperSco stars w/ discs in WISEA with SpT M0-G2 (single): 11
Number of UpperSco stars in WISEA with SpT M4-G0 (both): 334
Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (both): 43
Number of UpperSco stars in WISEA with SpT M4-G0 (single): 333
Number of UpperSco stars w/ discs in WISEA with SpT M4-G0 (single): 42
Number of UpperSco stars in WISEA with SpT M8-M4 (both): 674
Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (both): 111
Number of UpperSco stars in WISEA with SpT M8-M4 (single): 674
Number of UpperSco stars w/ discs in WISEA with SpT M8-M4 (single): 111


In [23]:
# Single targets:
unique_lupo_k = matched_lupo_k[~matched_lupo_k["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_lupo_k   = matched_lupo_k[ matched_lupo_k["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_lupo_k)+len(dups_lupo_k)}")
print(f"Length duplicates: {len(np.unique(dups_lupo_k["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_lupo_k_discless    = dups_lupo_k[ dups_lupo_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_lupo_k_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_lupo_k_discbearing = dups_lupo_k[~dups_lupo_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_lupo_k_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_lupo_k_discbearing[dups_lupo_k_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
lupo_k_both = pd.concat([unique_lupo_k, dups_lupo_k_discless, dups_lupo_k_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(lupo_k_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
lupo_k_sing = pd.concat([unique_lupo_k, dups_lupo_k_discless, dups_lupo_k_discbearing[dups_lupo_k_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(lupo_k_sing)}")

# Select disc-bearing stars in each catalogue
lupo_kDB_both = lupo_k_both[~lupo_k_both["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs both  : {len(lupo_kDB_both)}")
lupo_kDB_sing = lupo_k_sing[~lupo_k_sing["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs single: {len(lupo_kDB_sing)}")

Check that length unique+duplicates matches table: 860
Length duplicates: 16
Length duplicates disc less   : 11
Length duplicates disc bearing: 5
Length duplicates closest disc bearing: 5
Number of stars both  : 860
Number of stars single: 855
Number of stars w/ discs both  : 185
Number of stars w/ discs single: 180


In [24]:
print(
    "Number of Lupus stars in WISEA with SpT M0-G2 (both):", 
    len(lupo_k_both[lupo_k_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (both):", 
    len(lupo_kDB_both[lupo_kDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of Lupus stars in WISEA with SpT M0-G2 (single):", 
    len(lupo_k_sing[lupo_k_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (single):", 
    len(lupo_kDB_sing[lupo_kDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)

print(
    "Number of Lupus stars in WISEA with SpT M4-G0 (both):", 
    len(lupo_k_both[lupo_k_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (both):", 
    len(lupo_kDB_both[lupo_kDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of Lupus stars in WISEA with SpT M4-G0 (single):", 
    len(lupo_k_sing[lupo_k_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (single):", 
    len(lupo_kDB_sing[lupo_kDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)

print(
    "Number of Lupus stars in WISEA with SpT M8-M4 (both):", 
    len(lupo_k_both[lupo_k_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (both):", 
    len(lupo_kDB_both[lupo_kDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of Lupus stars in WISEA with SpT M8-M4 (single):", 
    len(lupo_k_sing[lupo_k_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (single):", 
    len(lupo_kDB_sing[lupo_kDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)

Number of Lupus stars in WISEA with SpT M0-G2 (both): 54
Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (both): 20
Number of Lupus stars in WISEA with SpT M0-G2 (single): 54
Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (single): 20
Number of Lupus stars in WISEA with SpT M4-G0 (both): 87
Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (both): 28
Number of Lupus stars in WISEA with SpT M4-G0 (single): 87
Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (single): 28
Number of Lupus stars in WISEA with SpT M8-M4 (both): 273
Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (both): 83
Number of Lupus stars in WISEA with SpT M8-M4 (single): 271
Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (single): 81


In [25]:
# Single targets:
unique_lupo_kp = matched_lupo_kp[~matched_lupo_kp["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_lupo_kp   = matched_lupo_kp[ matched_lupo_kp["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_lupo_kp)+len(dups_lupo_kp)}")
print(f"Length duplicates: {len(np.unique(dups_lupo_kp["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_lupo_kp_discless    = dups_lupo_kp[ dups_lupo_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_lupo_kp_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_lupo_kp_discbearing = dups_lupo_kp[~dups_lupo_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_lupo_kp_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_lupo_kp_discbearing[dups_lupo_kp_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
lupo_kp_both = pd.concat([unique_lupo_kp, dups_lupo_kp_discless, dups_lupo_kp_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(lupo_kp_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
lupo_kp_sing = pd.concat([unique_lupo_kp, dups_lupo_kp_discless, dups_lupo_kp_discbearing[dups_lupo_kp_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(lupo_kp_sing)}")

# Select disc-bearing stars in each catalogue
lupo_kpDB_both = lupo_kp_both[~lupo_kp_both["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs both  : {len(lupo_kpDB_both)}")
lupo_kpDB_sing = lupo_kp_sing[~lupo_kp_sing["DiskType Disk Typec"].isin(excluded_class)]
print(f"Number of stars w/ discs single: {len(lupo_kpDB_sing)}")

Check that length unique+duplicates matches table: 119
Length duplicates: 5
Length duplicates disc less   : 2
Length duplicates disc bearing: 3
Length duplicates closest disc bearing: 3
Number of stars both  : 119
Number of stars single: 116
Number of stars w/ discs both  : 76
Number of stars w/ discs single: 73


In [26]:
print(
    "Number of Lupus stars in WISEA with SpT M0-G2 (both):", 
    len(lupo_kp_both[lupo_kp_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (both):", 
    len(lupo_kpDB_both[lupo_kpDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of Lupus stars in WISEA with SpT M0-G2 (single):", 
    len(lupo_kp_sing[lupo_kp_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (single):", 
    len(lupo_kpDB_sing[lupo_kpDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM0G2)),na=False)])
)

print(
    "Number of Lupus stars in WISEA with SpT M4-G0 (both):", 
    len(lupo_kp_both[lupo_kp_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (both):", 
    len(lupo_kpDB_both[lupo_kpDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of Lupus stars in WISEA with SpT M4-G0 (single):", 
    len(lupo_kp_sing[lupo_kp_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (single):", 
    len(lupo_kpDB_sing[lupo_kpDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM4M0)),na=False)])
)

print(
    "Number of Lupus stars in WISEA with SpT M8-M4 (both):", 
    len(lupo_kp_both[lupo_kp_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (both):", 
    len(lupo_kpDB_both[lupo_kpDB_both["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of Lupus stars in WISEA with SpT M8-M4 (single):", 
    len(lupo_kp_sing[lupo_kp_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)
print(
    "Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (single):", 
    len(lupo_kpDB_sing[lupo_kpDB_sing["AdoptSpType"].str.contains("|".join(map(re.escape, maskM8M4)),na=False)])
)

Number of Lupus stars in WISEA with SpT M0-G2 (both): 17
Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (both): 13
Number of Lupus stars in WISEA with SpT M0-G2 (single): 17
Number of Lupus stars w/ discs in WISEA with SpT M0-G2 (single): 13
Number of Lupus stars in WISEA with SpT M4-G0 (both): 27
Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (both): 20
Number of Lupus stars in WISEA with SpT M4-G0 (single): 27
Number of Lupus stars w/ discs in WISEA with SpT M4-G0 (single): 20
Number of Lupus stars in WISEA with SpT M8-M4 (both): 65
Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (both): 36
Number of Lupus stars in WISEA with SpT M8-M4 (single): 63
Number of Lupus stars w/ discs in WISEA with SpT M8-M4 (single): 34


`Task 4:` Number of k and kp member in the catalogue of Carpenter+25 with a COflux or a COsize.

In [27]:
# Load your extended version of the table:
uppersco_mine = pd.read_csv(
    "uppersco_vs_polnitzky.dat",
    names  = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "discClassC25", "discClassL22", "COsizeS21?", "comments"
    ],
    sep      = r"\s+",
    skiprows = 1
)
print(len(uppersco_mine))
# Numbers match: 284.

284


In [28]:
# Update the SpT column in such a way to fill in blank spaces in the PPVII with those from Luhman (2022a)
uppersco_mine["SpT_E26"] = uppersco_mine["SpT_E26"].mask(uppersco_mine["SpT_E26"] == "-",uppersco_mine["SpT_L22"])
# uppersco_mine["SpT_E26"].tolist()

In [29]:
uppersco_mine_DB = uppersco_mine[~uppersco_mine["discClassL22"].isin(excluded_class)]

In [30]:
print(
    f"Stars with discs targeted in UpperSco K:",
    len(uppersco_mine_DB[ uppersco_mine_DB["memK_L22" ]=='Y'])
)
print(
    f"Stars with discs targeted in UpperSco K with SpT M0-G2:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco K with SpT M4-M0:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco K with SpT M8-M4:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

print(
    f"Stars with discs targeted in UpperSco KP:", 
    len(uppersco_mine_DB[ uppersco_mine_DB["memKP_L22"]=='Y'])
)
print(
    f"Stars with discs targeted in UpperSco KP with SpT M0-G2 :", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco KP with SpT M4-M0 :", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco KP with SpT M8-M4 :", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

Stars with discs targeted in UpperSco K: 164
Stars with discs targeted in UpperSco K with SpT M0-G2: 13
Stars with discs targeted in UpperSco K with SpT M4-M0: 50
Stars with discs targeted in UpperSco K with SpT M8-M4: 100
Stars with discs targeted in UpperSco KP: 140
Stars with discs targeted in UpperSco KP with SpT M0-G2 : 11
Stars with discs targeted in UpperSco KP with SpT M4-M0 : 42
Stars with discs targeted in UpperSco KP with SpT M8-M4 : 87


In [31]:
print(
    f"Stars with discs resolved in UpperSco K:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y')])
)
print(
    f"Stars with discs targeted in UpperSco K with SpT M0-G2:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco K with SpT M4-M0:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco K with SpT M8-M4:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

print(
    f"Stars with discs resolved in UpperSco KP:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y')])
)
print(
    f"Stars with discs targeted in UpperSco KP with SpT M0-G2:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco KP with SpT M4-M0:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in UpperSco KP with SpT M8-M4:", 
    len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y') & (uppersco_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

Stars with discs resolved in UpperSco K: 69
Stars with discs targeted in UpperSco K with SpT M0-G2: 13
Stars with discs targeted in UpperSco K with SpT M4-M0: 30
Stars with discs targeted in UpperSco K with SpT M8-M4: 26
Stars with discs resolved in UpperSco KP: 62
Stars with discs targeted in UpperSco KP with SpT M0-G2: 11
Stars with discs targeted in UpperSco KP with SpT M4-M0: 27
Stars with discs targeted in UpperSco KP with SpT M8-M4: 24


In [32]:
lupus_mine = pd.read_csv(
    "lupus_vs_polnitzky.dat",
    skiprows = 47,
    sep      = r'\s+',
    names   = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "mem_T22", "discClassL22", "COsizeS21?", "comments"
    ]
)
print(len(lupus_mine))
# Numbers match: 100, yes!

99


In [33]:
# Update the SpT column in such a way to fill in blank spaces in the PPVII with those from Luhman (2022a)
lupus_mine["SpT_E26"] = lupus_mine["SpT_E26"].mask(lupus_mine["SpT_E26"] == "-",lupus_mine["SpT_L22"])
# lupus_mine["SpT_E26"].tolist()

In [34]:
# The following Lupus targets are debris/evolved tr in Luhman (2022b) but have detected continuum/CO and resolved gas with ALMA. 
# Will update them to count as full discs (just to be properly considered in the counting)
# 2MASS J15580252-3736026 WISEA J155802.50-373602.9
# 2MASS J16071159-3903475 WISEA J160711.59-390347.7
# 2MASS J16085468-3937431 WISEA J160854.67-393743.3
# 2MASS J16083733-3923109 WISEA J160854.67-393743.3

print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J15580252-3736026')&(lupus_mine["SourceID_WISE"]=='J155802.50-373602.9')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16071159-3903475')&(lupus_mine["SourceID_WISE"]=='J160711.59-390347.7')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16085468-3937431')&(lupus_mine["SourceID_WISE"]=='J160854.67-393743.3')])
# print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16083733-3923109')])

mask = (
    ((lupus_mine["SourceID_2MASS"] == "J15580252-3736026") & (lupus_mine["SourceID_WISE"] == "J155802.50-373602.9")) |
    ((lupus_mine["SourceID_2MASS"] == "J16071159-3903475") & (lupus_mine["SourceID_WISE"] == "J160711.59-390347.7")) |
    ((lupus_mine["SourceID_2MASS"] == "J16085468-3937431") & (lupus_mine["SourceID_WISE"] == "J160854.67-393743.3"))
    # ((lupus_mine["SourceID_2MASS"] == "J16083733-3923109")                                                      )
)
# Inspect the lupus_mine rows:
# print(lupus_mine.loc[mask])     
lupus_mine.loc[mask,"discClassL22"] = "full_riclassified"

print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J15580252-3736026')&(lupus_mine["SourceID_WISE"]=='J155802.50-373602.9')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16071159-3903475')&(lupus_mine["SourceID_WISE"]=='J160711.59-390347.7')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16085468-3937431')&(lupus_mine["SourceID_WISE"]=='J160854.67-393743.3')])
# print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16083733-3923109')])

21    debris/ev-trans
Name: discClassL22, dtype: object
38    debris/ev-trans
Name: discClassL22, dtype: object
65    debris/ev-trans
Name: discClassL22, dtype: object
21    full_riclassified
Name: discClassL22, dtype: object
38    full_riclassified
Name: discClassL22, dtype: object
65    full_riclassified
Name: discClassL22, dtype: object


In [35]:
lupus_mine_DB = lupus_mine[~lupus_mine["discClassL22"].isin(excluded_class)]

In [36]:
print(
    f"Stars with discs targeted in Lupus K:",
    len(lupus_mine_DB[ lupus_mine_DB["memK_L22" ]=='Y'])
)
print(
    f"Stars with discs targeted in Lupus K with SpT M0-G2:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus K with SpT M4-M0:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus K with SpT M8-M4:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

print(
    f"Stars with discs targeted in Lupus KP:", 
    len(lupus_mine_DB[ lupus_mine_DB["memKP_L22"]=='Y'])
)
print(
    f"Stars with discs targeted in Lupus KP with SpT M0-G2 :", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus KP with SpT M4-M0 :", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus KP with SpT M8-M4 :", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

Stars with discs targeted in Lupus K: 61
Stars with discs targeted in Lupus K with SpT M0-G2: 13
Stars with discs targeted in Lupus K with SpT M4-M0: 17
Stars with discs targeted in Lupus K with SpT M8-M4: 29
Stars with discs targeted in Lupus KP: 57
Stars with discs targeted in Lupus KP with SpT M0-G2 : 12
Stars with discs targeted in Lupus KP with SpT M4-M0 : 14
Stars with discs targeted in Lupus KP with SpT M8-M4 : 29


In [37]:
print(
    f"Stars with discs resolved in Lupus K:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y')])
)
print(
    f"Stars with discs targeted in Lupus K with SpT M0-G2:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus K with SpT M4-M0:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus K with SpT M8-M4:", 
    len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

print(
    f"Stars with discs resolved in Lupus KP:", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y')])
)
print(
    f"Stars with discs targeted in Lupus KP with SpT M0-G2:", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM0G2)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus KP with SpT M4-M0:", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM4M0)),na=False))])
)
print(
    f"Stars with discs targeted in Lupus KP with SpT M8-M4:", 
    len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y') & (lupus_mine_DB["SpT_E26"].str.contains("|".join(map(re.escape, maskM8M4)),na=False))])
)

Stars with discs resolved in Lupus K: 43
Stars with discs targeted in Lupus K with SpT M0-G2: 11
Stars with discs targeted in Lupus K with SpT M4-M0: 13
Stars with discs targeted in Lupus K with SpT M8-M4: 18
Stars with discs resolved in Lupus KP: 39
Stars with discs targeted in Lupus KP with SpT M0-G2: 10
Stars with discs targeted in Lupus KP with SpT M4-M0: 10
Stars with discs targeted in Lupus KP with SpT M8-M4: 18
